# Notebook 006: Date & Timestamp Mastery: Parsing, Arithmetic & Timezones

**Module:** Module 01: Core PySpark Foundations & DataFrame API  
**Difficulty:** `Beginner`  
**Databricks Masterclass Reference:** Video 17: Lakeflow Declarative Pipelines & Video 20: Finance Domain E2E Project  

---

## 1. Business Scenario & Real-World Context
A global ride-sharing platform records ride dispatch, pickup, and dropoff events across multiple international timezones. Operations teams require accurate trip duration in minutes, local business hour classifications, and SLA reporting.

## 2. Technical Requirement & Problem Statement
Master datetime transformations: parse timestamps with explicit timezone offsets, convert between UTC and local timezones (`from_utc_timestamp`), perform date arithmetic (`date_add`, `datediff`), and compute precise duration intervals.

## 3. Spark & Databricks Architecture Notes
Spark 3.0 introduced Proleptic Gregorian calendar support by default (`spark.sql.session.timeZone`). Spark stores timestamps internally as UTC epoch microseconds. Using timezone-aware functions ensures consistency across distributed worker nodes regardless of server OS timezone settings.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_006_Date_&_Timestamp_Mastery_Parsing,_Arithmetic_&_Timezones")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Mocking Global Ride Events
Generate ride event timestamps with varied timezones.

In [ ]:
rides_data = [
    ("RIDE-01", "2023-11-15T08:15:00Z", "2023-11-15T08:42:30Z", "America/New_York"),
    ("RIDE-02", "2023-11-15T14:30:00Z", "2023-11-15T15:05:00Z", "Europe/London"),
    ("RIDE-03", "2023-11-15T22:45:00Z", "2023-11-15T23:25:15Z", "Asia/Tokyo"),
    ("RIDE-04", "2023-11-16T01:10:00Z", "2023-11-16T01:35:00Z", "America/Los_Angeles")
]

df_rides = spark.createDataFrame(rides_data, ["ride_id", "pickup_utc", "dropoff_utc", "tz_city"])
df_rides.show(truncate=False)

## 6. Parsing Timestamps & Computing Duration
Convert ISO strings to native timestamps and compute duration in seconds and fractional minutes.

In [ ]:
df_parsed = df_rides.withColumn(
    "pickup_ts", F.to_timestamp(F.col("pickup_utc"))
).withColumn(
    "dropoff_ts", F.to_timestamp(F.col("dropoff_utc"))
)

# Compute duration using unix timestamp difference
df_duration = df_parsed.withColumn(
    "duration_seconds",
    F.col("dropoff_ts").cast(T.LongType()) - F.col("pickup_ts").cast(T.LongType())
).withColumn(
    "duration_minutes",
    F.round(F.col("duration_seconds") / 60.0, 2)
)

df_duration.select("ride_id", "pickup_ts", "dropoff_ts", "duration_minutes").show()

## 7. Converting to Local Time & Extracting Temporal Dimensions
Translate UTC timestamps to local city timezones and derive reporting dimensions.

In [ ]:
df_local = df_duration.withColumn(
    "pickup_local",
    F.from_utc_timestamp(F.col("pickup_ts"), F.col("tz_city"))
).withColumn(
    "pickup_hour",
    F.hour(F.col("pickup_local"))
).withColumn(
    "day_of_week",
    F.date_format(F.col("pickup_local"), "EEEE")
).withColumn(
    "is_peak_hour",
    F.when(F.col("pickup_hour").between(7, 9) | F.col("pickup_hour").between(17, 19), True).otherwise(False)
)

df_local.select("ride_id", "tz_city", "pickup_local", "pickup_hour", "day_of_week", "is_peak_hour").show(truncate=False)

## 9. Key Architectural Takeaways
- **Always store raw ingestion timestamps in UTC to prevent daylight saving ambiguities.**
- **Use `from_utc_timestamp` and `to_utc_timestamp` for timezone shifts rather than manual hour additions.**
- **Calculate sub-minute precision by casting timestamps to `LongType` (epoch seconds) or using `unix_timestamp`.**

## 10. Senior Data Engineering Interview Questions
1. How does Apache Spark handle daylight saving time transitions?
2. What changed between the Legacy Julian/Gregorian calendar and Spark 3.0 Proleptic Gregorian calendar?
3. How do you calculate the business days between two dates in PySpark without a UDF?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
